# Run one VQE Example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from typing import Sequence
import pyscf



from qiskit import QuantumCircuit, QuantumRegister

from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.base import BaseEstimatorV2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, Estimator

import ffsim

In [ ]:
import pyscf

# Build H2 molecule
mol = pyscf.gto.Mole()
mol.build(atom=[["H", (0, 0, 0)], ["H", (1.0, 0, 0)]], basis="sto-3g", symmetry="Dooh")

# Get molecular data and Hamiltonian
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf)
mol_hamiltonian = mol_data.hamiltonian

# Convert MolecularHamiltonian to FermionOperator
ferm_op = ffsim.fermion_operator(mol_hamiltonian)

Hamiltonian = ffsim.qiskit.jordan_wigner(ferm_op)


# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())
 
# Get molecular integrals
num_orbitals = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

CASCI = cas.run()

In [ ]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

In [ ]:
def BuildCircuit(T1Tensor,T2Tensor,Layers):
    alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
    alpha_beta_indices = [(p, p) for p in range(0, num_orbitals, 4)]
     
     
    ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
        t2=T2Tensor,
        t1=T1Tensor,
        n_reps=Layers,
        interaction_pairs=(alpha_alpha_indices, alpha_beta_indices)
    )
    
    nelec = (num_elec_a, num_elec_b)
     
    # create an empty quantum circuit
    qubits = QuantumRegister(2 * num_orbitals, name="q")
    circuit = QuantumCircuit(qubits)
     
    # prepare Hartree-Fock state as the reference state and append it to the quantum circuit
    circuit.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)
     
    # apply the UCJ operator to the reference state
    circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
    # circuit.measure_all()
    return circuit

In [ ]:
def cost_func(
    x, 
    t1_shape, 
    t2_shape, 
    t1_size,
    Layers,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
) -> float:
    """Ground state energy evaluation."""
    T1Tensor = x[:t1_size].reshape(t1_shape)
    T2Tensor = x[t1_size:].reshape(t2_shape)
    
    ansatz=BuildCircuit(T1Tensor,T2Tensor,Layers)
    return (
        estimator.run([(ansatz, hamiltonian)])
        .result()[0]
        .data.evs
    )
    
def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Iteration")
    plt.ylabel("Energy")
    plt.show()


def build_callback(t1_shape, t2_shape, t1_size,Layers, 
                   circuit, hamiltonian, estimator, callback_dict):
    def callback(x):
        T1Tensor = x[:t1_size].reshape(t1_shape)
        T2Tensor = x[t1_size:].reshape(t2_shape)
        
        ansatz=BuildCircuit(T1Tensor,T2Tensor,Layers)        
        # Keep track of the number of iterations
        callback_dict["iters"] += 1
        # Compute the value of the cost function at the current vector
        current_cost = (
            estimator.run([(ansatz, hamiltonian)])
            .result()[0]
            .data.evs
        )
        callback_dict["cost_history"].append(current_cost)
        # Print to screen on single line
        print(
            "Iters. done: {} [Current cost: {}]".format(
                callback_dict["iters"], current_cost
            ),
            end="\r",
            flush=True,
        )

    return callback

In [ ]:
# Generate a pass manager without providing a backend
from qiskit.transpiler import generate_preset_pass_manager
circuit = BuildCircuit(t1,t2,1)

pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
isa_Hamiltonian = Hamiltonian.apply_layout(isa_circuit.layout)
from qiskit.primitives import StatevectorEstimator
 
estimator = StatevectorEstimator()

job = estimator.run([(circuit, Hamiltonian)])
result = job.result()
print(f" > Result class: {type(result)}")


print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")



In [ ]:
t1_shape=t1.shape
t2_shape=t2.shape
t1_size=t1.size
Layers=10


x0=np.hstack([t1.flatten(),t2.flatten()])



In [ ]:
callback_dict = {
    "iters": 0,
    "cost_history": [],
}

callback = build_callback(t1_shape, t2_shape, t1_size,Layers,
    circuit, Hamiltonian, estimator, callback_dict
)
res = minimize(
    cost_func,
    x0=x0,
    args=(t1_shape, t2_shape, t1_size,Layers, Hamiltonian, estimator),
    method="cobyla",
    callback=callback
    # options={"maxiter": 100},
)

visualize_results(callback_dict)


In [ ]:
plt.hlines(scf.e_tot,0,300,color='g',linestyle='-.')
plt.hlines(CASCI.e_tot,0,300,color='g',linestyle='-.')
plt.hlines(ccsd.e_tot,0,300,color='r',linestyles='--')
plt.plot(callback_dict["cost_history"], lw=2)
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.show()